# Adding Memory Layer

## 1. Import necessary packages

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

## 2. Instantiate OpenAI client

In [2]:
load_dotenv()

client = OpenAI()

## 3. Define Parameters

In [3]:
model = "gpt-5-nano-2025-08-07"

temperature = 1.0

## 4. Define User prompt

In [4]:
system_prompt = "You are a helpful assistant."
user_prompt = "What was my previous question?"

In [5]:
messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

response.choices[0].message.content

'I can’t see your previous question. I only see the current message. If you tell me what it was or paste it here, I’ll help with it. You can also scroll up in the chat to view your earlier message.'

## 5. Define Memory

In [7]:
memory = [
    {"role": "system", "content": "You're a Senior React Developer."},
    {"role": "user", "content": "What is useEffect in React? Answer in brief."},
]

In [8]:
new_response = client.chat.completions.create(
        model=model,
        messages=memory,
        temperature=temperature,
    )

memory.append({"role": "assistant", "content": new_response.choices[0].message.content})

memory

[{'role': 'system', 'content': "You're a Senior React Developer."},
 {'role': 'user', 'content': 'What is useEffect in React? Answer in brief.'},
 {'role': 'assistant',
  'content': 'useEffect is a React Hook that lets you perform side effects in function components. It runs after render and can clean up before the next effect or on unmount. The effect’s execution is controlled by a dependencies array:\n\n- Without dependencies: runs after every render.\n- With []: runs once after the initial render (mount).\n- With [dep1, dep2]: runs after initial render and whenever any listed dependency changes.\n\nCommon uses: data fetching, subscriptions, manual DOM manipulation, logging. If you return a function from the effect, it’s used for cleanup. Example:\n\nuseEffect(() => {\n  document.title = `Count ${count}`;\n  return () => { /* cleanup */ };\n}, [count]);'}]

In [9]:
memory.append({"role": "user", "content": "What was my previous question?"})

memory

[{'role': 'system', 'content': "You're a Senior React Developer."},
 {'role': 'user', 'content': 'What is useEffect in React? Answer in brief.'},
 {'role': 'assistant',
  'content': 'useEffect is a React Hook that lets you perform side effects in function components. It runs after render and can clean up before the next effect or on unmount. The effect’s execution is controlled by a dependencies array:\n\n- Without dependencies: runs after every render.\n- With []: runs once after the initial render (mount).\n- With [dep1, dep2]: runs after initial render and whenever any listed dependency changes.\n\nCommon uses: data fetching, subscriptions, manual DOM manipulation, logging. If you return a function from the effect, it’s used for cleanup. Example:\n\nuseEffect(() => {\n  document.title = `Count ${count}`;\n  return () => { /* cleanup */ };\n}, [count]);'},
 {'role': 'user', 'content': 'What was my previous question?'}]

In [10]:
new_response = client.chat.completions.create(
        model=model,
        messages=memory,
        temperature=temperature,
    )

memory.append({"role": "assistant", "content": new_response.choices[0].message.content})

memory

[{'role': 'system', 'content': "You're a Senior React Developer."},
 {'role': 'user', 'content': 'What is useEffect in React? Answer in brief.'},
 {'role': 'assistant',
  'content': 'useEffect is a React Hook that lets you perform side effects in function components. It runs after render and can clean up before the next effect or on unmount. The effect’s execution is controlled by a dependencies array:\n\n- Without dependencies: runs after every render.\n- With []: runs once after the initial render (mount).\n- With [dep1, dep2]: runs after initial render and whenever any listed dependency changes.\n\nCommon uses: data fetching, subscriptions, manual DOM manipulation, logging. If you return a function from the effect, it’s used for cleanup. Example:\n\nuseEffect(() => {\n  document.title = `Count ${count}`;\n  return () => { /* cleanup */ };\n}, [count]);'},
 {'role': 'user', 'content': 'What was my previous question?'},
 {'role': 'assistant',
  'content': 'Your previous question was: 

## 7. Create Memory Class

In [11]:
from typing import List, Dict, Literal

class Memory:
    def __init__(self):
        self.messages: List[Dict[str, str]] = []
    
    def add_message(self, role: Literal['user', 'system', 'assistant'], content: str):
        self.messages.append({
            "role": role,
            "content": content
        })

    def get_messages(self) -> List[Dict[str, str]]:
        return self.messages

## 8. Create Helper chat function

In [ ]:
def chat(user_message:str=None, memory:Memory=None)->str:
    messages = [{"role": "user", "content": user_message}]
    
    if memory:
        if user_message:
            memory.add_message(role="user", content=user_message)
        messages = memory.get_messages()        

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

    ai_message = response.choices[0].message.content
    
    if memory:
        memory.add_message(role="assistant", content=ai_message)
    
    return ai_message

In [13]:
memory = Memory()

In [14]:
memory.add_message(role="system", content="You're a Senior React Developer. You don't know anything about other programming languages, so don't provide answers about languages like Java.")

In [15]:
memory.get_messages()

[{'role': 'system',
  'content': "You're a Senior React Developer. You don't know anything about other programming languages, so don't provide answers about languages like Java."}]

In [16]:
chat(
    user_message = "What is useEffect in React? Answer in brief.",
)

'useEffect is a React hook for running side effects in function components (e.g., data fetching, subscriptions, DOM mutations). It runs after render and can clean up with a return function. You pass an effect function and an optional dependency array:\n\n- If no dependencies are provided, it runs after every render.\n- If an empty array [] is provided, it runs once on mount.\n- If [dep1, dep2] is provided, it runs when any dependency changes.\n\nExample: useEffect(() => { document.title = `Count: ${count}`; return () => { /* cleanup */ }; }, [count]);'

In [17]:
memory.get_messages()

[{'role': 'system',
  'content': "You're a Senior React Developer. You don't know anything about other programming languages, so don't provide answers about languages like Java."}]

In [18]:
chat(
    user_message = "What is useEffect in React? Answer in brief.",
    memory=memory
)

'- useEffect is a React hook that lets you run side effects in function components after rendering.\n- It can run on mount, on updates (when specified dependencies change), and it can clean up on unmount.\n- Syntax: useEffect(() => { /* effect */ return () => { /* cleanup */ }; }, [dependencies]);\n- If you pass an empty array [], it runs once on mount. If you omit the array, it runs after every render.\n- Common uses: data fetching, subscriptions, timers, manually updating the DOM.'

In [19]:
chat(
    user_message = "What was my previous question?",
    memory=memory
)

'Your previous question was: "What is useEffect in React? Answer in brief."'

In [20]:
memory.get_messages()

[{'role': 'system',
  'content': "You're a Senior React Developer. You don't know anything about other programming languages, so don't provide answers about languages like Java."},
 {'role': 'user', 'content': 'What is useEffect in React? Answer in brief.'},
 {'role': 'assistant',
  'content': '- useEffect is a React hook that lets you run side effects in function components after rendering.\n- It can run on mount, on updates (when specified dependencies change), and it can clean up on unmount.\n- Syntax: useEffect(() => { /* effect */ return () => { /* cleanup */ }; }, [dependencies]);\n- If you pass an empty array [], it runs once on mount. If you omit the array, it runs after every render.\n- Common uses: data fetching, subscriptions, timers, manually updating the DOM.'},
 {'role': 'user', 'content': 'What was my previous question?'},
 {'role': 'assistant',
  'content': 'Your previous question was: "What is useEffect in React? Answer in brief."'}]